<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-DeepLearning/blob/main/mnist_CNN_vs_ViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-DeepLearning/blob/main/Beispiele/mnist_CNN_vs_ViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNIST: CNN gegen Vision Transformer

Dieses Notebook trainiert **zwei Modelle auf exakt denselben Daten**, mit demselben
Trainingsloop, demselben Optimierer und demselben "Budget" an Epochen:

| | Modell A | Modell B |
|---|---|---|
| Architektur | Convolutional Neural Network | Vision Transformer |
| Baustein | Faltung | Self-Attention |
| Annahme über die Daten | Nachbarschaft zählt, Position ist egal | keine — alles darf auf alles schauen |

Der einzige Unterschied zwischen den beiden Läufen ist die Architektur. Alles andere ist
identisch, damit der Vergleich etwas aussagt.

**Worauf es hinausläuft:** Der Transformer bringt *weniger* eingebautes Vorwissen mit als das
CNN. Was das für die Genauigkeit bedeutet, rechnest du gleich selbst nach — und das Ergebnis
ist der eigentliche Lerninhalt dieses Notebooks.

> Das CNN ist dasselbe wie in `mnist_CNN.ipynb`. Der Optimierer wurde für beide Modelle
> vereinheitlicht (AdamW statt Adadelta), sonst wäre der Vergleich verfälscht.

---
## 0 · Vorbereitung

In [ ]:
#@title Abhängigkeiten und Gerät
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    !pip install -q torch torchvision matplotlib tqdm --upgrade

import math, random, time
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Gerät:", device)

# Einheitliche Darstellung für alle Abbildungen
BLAU, ORANGE = "#1F6FB8", "#E8730C"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
})

In [ ]:
#@title Konfiguration — hier kannst du experimentieren
@dataclass
class Config:
    batch_size: int = 128
    test_batch_size: int = 512
    epochs: int = 5            # gleiches Budget für beide Modelle
    lr: float = 1e-3
    weight_decay: float = 1e-4
    val_size: int = 10_000
    num_workers: int = 2
    train_subset: int = 0      # 0 = alle 50'000. Kleiner Wert = weniger Daten (siehe Aufgabe 3)

cfg = Config()
print(cfg)

---
## 1 · Daten — einmal laden, beide Modelle teilen sie sich

Train/Validation/Test wie im CNN-Notebook: 50 000 / 10 000 / 10 000. Mit demselben Dataloader erhalten die Modelle auch dieselben Daten in der gleichen Reihenfolge.

In [ ]:
#@title MNIST laden und aufteilen
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

root = "./data"
full_train = datasets.MNIST(root, train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root, train=False, download=True, transform=transform)

train_size = len(full_train) - cfg.val_size
train_set, val_set = random_split(
    full_train, [train_size, cfg.val_size],
    generator=torch.Generator().manual_seed(SEED),
)

if cfg.train_subset:                      # optional: Training künstlich verknappen
    train_set = torch.utils.data.Subset(train_set, range(cfg.train_subset))

loader_args = dict(num_workers=cfg.num_workers, pin_memory=(device.type == "cuda"))
train_loader = DataLoader(train_set, batch_size=cfg.batch_size, shuffle=True, **loader_args)
val_loader   = DataLoader(val_set,   batch_size=cfg.test_batch_size, shuffle=False, **loader_args)
test_loader  = DataLoader(mnist_test, batch_size=cfg.test_batch_size, shuffle=False, **loader_args)

print(f"Training   {len(train_set):>6,}")
print(f"Validation {len(val_set):>6,}")
print(f"Test       {len(mnist_test):>6,}")

---
## 2 · Modell A — das CNN

Unverändert aus `mnist_CNN.ipynb`. Zwei Faltungsschichten, Max-Pooling, zwei vollverbundene
Schichten. Das Entscheidende steckt nicht im Code, sondern in den Annahmen, die er trifft:

- **Lokalität** — ein 3×3-Kernel schaut nur auf benachbarte Pixel. Dass zusammengehörige
  Bildinformation räumlich beieinander liegt, muss das Netz nicht lernen, es ist eingebaut.
- **Gewichtsteilung** — derselbe Kernel läuft über das ganze Bild. Eine Kante oben links wird
  mit denselben Parametern erkannt wie eine unten rechts.

Dieses eingebaute Vorwissen heisst **induktiver Bias**. Es verkleinert den Raum der Funktionen,
die das Netz überhaupt darstellen kann — und genau deshalb lernt es schneller.

In [ ]:
#@title CNN
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout2d(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(64 * 12 * 12, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))        # (B, 32, 26, 26)
        x = F.relu(self.conv2(x))        # (B, 64, 24, 24)
        x = F.max_pool2d(x, 2)           # (B, 64, 12, 12)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        return self.fc2(x)               # Logits


def anzahl_parameter(modell):
    return sum(p.numel() for p in modell.parameters() if p.requires_grad)


print(f"CNN: {anzahl_parameter(CNN()):,} trainierbare Parameter")

---
## 3 · Modell B — der Vision Transformer

Ein Transformer kennt keine Bilder, er kennt nur **Sequenzen von Tokens**. Also machen wir aus
dem Bild eine Sequenz: Das 28×28-Bild wird in 16 Felder zu 7×7 Pixeln zerschnitten, jedes Feld
wird flachgeklopft und durch eine lineare Schicht auf einen Vektor der Länge `dim` abgebildet.

Das ist exakt dieselbe Operation wie die Embedding-Tabelle aus der Vorlesung — nur dass hier
nicht ein Wortindex nachgeschlagen, sondern ein Pixelblock projiziert wird. Danach weiss der
Transformer nicht mehr, dass er ein Bild vor sich hat.

Zwei Dinge kommen dazu:

- Ein **Positions-Embedding**, weil Self-Attention die Reihenfolge sonst nicht kennt. Ohne das
  wäre ein verwürfeltes Bild für das Modell dasselbe Bild.
- Ein **CLS-Token**, ein zusätzlicher gelernter Vektor am Anfang der Sequenz. Er gehört zu keinem
  Bildausschnitt, sammelt über die Attention-Schichten Information aus allen Patches ein, und
  nur aus ihm wird am Schluss die Klasse vorhergesagt.

In [ ]:
#@title Vom Bild zur Token-Sequenz — einmal anschauen
bild, ziel = mnist_test[0]
PATCH = 7
G = 28 // PATCH                                  # 4 x 4 = 16 Patches

patches = bild.squeeze(0).unfold(0, PATCH, PATCH).unfold(1, PATCH, PATCH)   # (4, 4, 7, 7)

fig = plt.figure(figsize=(8.4, 3.4))
links = fig.add_subplot(1, 2, 1)
links.imshow(bild.squeeze(0), cmap="gray")
links.set_title(f"Originalbild — Ziffer {ziel}", loc="left", fontsize=10)
links.set_xticks([]); links.set_yticks([])
for k in range(1, G):
    links.axhline(k * PATCH - 0.5, color=ORANGE, linewidth=1.2)
    links.axvline(k * PATCH - 0.5, color=ORANGE, linewidth=1.2)

rechts = fig.add_subplot(1, 2, 2)
rechts.set_title(f"{G*G} Patches à {PATCH}×{PATCH} = {G*G} Tokens", loc="left", fontsize=10)
rechts.set_xticks([]); RAND = 1.6
leinwand = np.zeros((G * (PATCH + 2), G * (PATCH + 2)))
for i in range(G):
    for j in range(G):
        y, x = i * (PATCH + 2), j * (PATCH + 2)
        leinwand[y:y + PATCH, x:x + PATCH] = patches[i, j]
rechts.imshow(leinwand, cmap="gray")
rechts.set_xticks([]); rechts.set_yticks([])
plt.tight_layout()
plt.show()

print("Ein Patch wird zu einem Vektor der Länge", PATCH * PATCH,
      "und dann linear auf dim projiziert.")

In [ ]:
#@title Vision Transformer
class TransformerBlock(nn.Module):
    """Ein Transformer-Block in der Pre-Norm-Variante:
    Norm -> Self-Attention -> Residual, dann Norm -> Feed-Forward -> Residual."""

    def __init__(self, dim, heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_dim, dim), nn.Dropout(dropout),
        )
        self.letzte_attention = None      # nur für die Visualisierung weiter unten
        self.attention_merken = False

    def forward(self, x):
        h = self.norm1(x)
        if self.attention_merken:
            a, gewichte = self.attn(h, h, h, need_weights=True, average_attn_weights=False)
            self.letzte_attention = gewichte.detach()        # (B, Köpfe, L, L)
        else:
            a, _ = self.attn(h, h, h, need_weights=False)
        x = x + a                                            # Residual 1
        x = x + self.mlp(self.norm2(x))                      # Residual 2
        return x


class ViT(nn.Module):
    def __init__(self, bildgroesse=28, patch=7, dim=64, tiefe=4, heads=4,
                 mlp_dim=128, num_classes=10, dropout=0.1):
        super().__init__()
        assert bildgroesse % patch == 0
        self.patch = patch
        self.gitter = bildgroesse // patch
        n_patches = self.gitter ** 2

        # 1. Patch-Embedding: jeder Pixelblock wird zu einem Vektor
        self.patch_embedding = nn.Linear(patch * patch, dim)

        # 2. CLS-Token und Positions-Embedding, beide gelernt
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embedding = nn.Parameter(torch.zeros(1, n_patches + 1, dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)
        self.dropout = nn.Dropout(dropout)

        # 3. Ein Stapel identisch gebauter Transformer-Blöcke
        self.bloecke = nn.ModuleList([
            TransformerBlock(dim, heads, mlp_dim, dropout) for _ in range(tiefe)
        ])

        # 4. Klassifikationskopf, nur auf dem CLS-Token
        self.norm = nn.LayerNorm(dim)
        self.kopf = nn.Linear(dim, num_classes)

    def zu_tokens(self, x):
        B = x.shape[0]
        p, g = self.patch, self.gitter
        x = x.squeeze(1).unfold(1, p, p).unfold(2, p, p)      # (B, g, g, p, p)
        x = x.reshape(B, g * g, p * p)                        # (B, n_patches, p*p)
        return x

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embedding(self.zu_tokens(x))           # (B, n_patches, dim)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)                        # (B, n_patches+1, dim)
        x = self.dropout(x + self.pos_embedding)

        for block in self.bloecke:
            x = block(x)

        return self.kopf(self.norm(x[:, 0]))                  # nur das CLS-Token


print(f"ViT: {anzahl_parameter(ViT()):,} trainierbare Parameter")

---
## 4 · Ein Trainingsloop für beide

Damit der Vergleich gilt, bekommen beide Modelle dieselbe Funktion: dieselben Loader, denselben
Optimierer, denselben Scheduler, dieselbe Anzahl Epochen. Gespeichert wird jeweils der Stand mit
der besten Validierungsgenauigkeit.

In [ ]:
#@title Training und Auswertung
@torch.no_grad()
def auswerten(modell, loader):
    modell.eval()
    richtig = gesamt = 0
    loss = 0.0
    for daten, ziel in loader:
        daten, ziel = daten.to(device), ziel.to(device)
        logits = modell(daten)
        loss += F.cross_entropy(logits, ziel, reduction="sum").item()
        richtig += (logits.argmax(dim=1) == ziel).sum().item()
        gesamt += ziel.numel()
    return loss / gesamt, 100.0 * richtig / gesamt


def trainieren(modell, name):
    modell = modell.to(device)
    optimierer = torch.optim.AdamW(modell.parameters(), lr=cfg.lr,
                                   weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimierer, T_max=cfg.epochs)

    verlauf = {"val_acc": [], "train_loss": []}
    beste_acc, bester_stand = 0.0, None
    start = time.time()

    for epoche in range(1, cfg.epochs + 1):
        modell.train()
        summe = n = 0
        fortschritt = tqdm(train_loader, desc=f"{name} | Epoche {epoche}/{cfg.epochs}",
                           leave=False)
        for daten, ziel in fortschritt:
            daten, ziel = daten.to(device), ziel.to(device)
            optimierer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(modell(daten), ziel)
            loss.backward()
            optimierer.step()
            summe += loss.item() * ziel.numel(); n += ziel.numel()
            fortschritt.set_postfix(loss=f"{summe / n:.4f}")
        scheduler.step()

        _, val_acc = auswerten(modell, val_loader)
        verlauf["train_loss"].append(summe / n)
        verlauf["val_acc"].append(val_acc)
        print(f"{name} | Epoche {epoche}/{cfg.epochs} | "
              f"Trainingsloss {summe / n:.4f} | Validierung {val_acc:.2f} %")

        if val_acc > beste_acc:
            beste_acc = val_acc
            bester_stand = {k: v.detach().cpu().clone() for k, v in modell.state_dict().items()}

    dauer = time.time() - start
    modell.load_state_dict(bester_stand)               # bester Stand laut Validierung
    _, test_acc = auswerten(modell, test_loader)

    ergebnis = {
        "name": name,
        "parameter": anzahl_parameter(modell),
        "dauer_s": dauer,
        "val_acc": beste_acc,
        "test_acc": test_acc,
        "verlauf": verlauf,
    }
    print(f"{name} fertig: beste Validierung {beste_acc:.2f} %, "
          f"Test {test_acc:.2f} %, {dauer:.0f} s\n")
    return modell, ergebnis

In [ ]:
#@title Beide Modelle trainieren
torch.manual_seed(SEED)
cnn, erg_cnn = trainieren(CNN(), "CNN")

torch.manual_seed(SEED)
vit, erg_vit = trainieren(ViT(), "ViT")

---
## 5 · Der Vergleich

In [ ]:
#@title Tabelle und Lernkurven
ergebnisse = [erg_cnn, erg_vit]

kopf = f"{'Modell':<8}{'Parameter':>14}{'Trainingszeit':>16}{'Validierung':>14}{'Test':>10}"
print(kopf)
print("-" * len(kopf))
for e in ergebnisse:
    print(f"{e['name']:<8}{e['parameter']:>14,}{e['dauer_s']:>14.0f} s"
          f"{e['val_acc']:>13.2f} %{e['test_acc']:>9.2f} %")

abstand = erg_cnn["test_acc"] - erg_vit["test_acc"]
print(f"\nAbstand auf dem Testset: {abstand:+.2f} Prozentpunkte zugunsten des "
      f"{'CNN' if abstand > 0 else 'ViT'}.")
print(f"Fehlerquote: CNN {100 - erg_cnn['test_acc']:.2f} %, ViT {100 - erg_vit['test_acc']:.2f} %")
print(f"Grössenverhältnis: das CNN hat "
      f"{erg_cnn['parameter'] / erg_vit['parameter']:.1f}-mal so viele Parameter wie der ViT.")

fig, ax = plt.subplots(figsize=(6.2, 3.4))
epochen = range(1, cfg.epochs + 1)
for e, farbe in zip(ergebnisse, [BLAU, ORANGE]):
    ax.plot(epochen, e["verlauf"]["val_acc"], color=farbe, linewidth=2,
            marker="o", markersize=5, label=e["name"])
    ax.annotate(f"{e['verlauf']['val_acc'][-1]:.1f} %",
                (cfg.epochs, e["verlauf"]["val_acc"][-1]), color=farbe,
                fontsize=9, fontweight="bold", xytext=(8, -3), textcoords="offset points")
ax.set_xlabel("Epoche"); ax.set_ylabel("Validierungsgenauigkeit in %")
ax.set_xticks(list(epochen))
ax.set_title("Lernkurven bei gleichem Budget", loc="left", fontsize=10)
ax.grid(axis="y", color="#E6E6E6", linewidth=1); ax.set_axisbelow(True)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print("Achte auf die Form der Kurven, nicht nur auf den Endwert:")
print("Flacht eine der beiden noch nicht ab? Dann war das Budget für sie zu knapp.")

In [ ]:
#@title Confusion Matrices nebeneinander
@torch.no_grad()
def confusion_matrix(modell, loader, num_classes=10):
    modell.eval()
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for daten, ziel in loader:
        daten = daten.to(device)
        pred = modell(daten).argmax(dim=1).cpu()
        for t, p in zip(ziel.view(-1), pred.view(-1)):
            cm[int(t), int(p)] += 1
    return cm


fig, achsen = plt.subplots(1, 2, figsize=(10.6, 4.6))
for ax, (modell, e) in zip(achsen, [(cnn, erg_cnn), (vit, erg_vit)]):
    cm = confusion_matrix(modell, test_loader)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    ax.set_xlabel("vorhergesagt"); ax.set_ylabel("tatsächlich")
    ax.set_title(f"{e['name']} — Test {e['test_acc']:.2f} %", loc="left", fontsize=10)
    for i in range(10):
        for j in range(10):
            if cm[i, j]:
                ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=6.5,
                        color="white" if cm_norm[i, j] > 0.5 else "#444444")
plt.tight_layout()
plt.show()

---
## 6 · Worauf schaut der Transformer?

Das CLS-Token sammelt die Information für die Klassifikation ein. Seine Attention-Gewichte im
letzten Block sagen also, **welche Bildausschnitte für die Entscheidung gezählt haben**. Weil es
16 Patches sind, ergibt das eine 4×4-Karte, die wir auf Bildgrösse hochskalieren.

Das ist dieselbe Idee wie die Interpretierbarkeitsmethoden aus Woche 7 — nur dass die Erklärung
hier direkt aus der Architektur fällt und nicht nachträglich berechnet werden muss.

In [ ]:
#@title Attention-Karte des CLS-Tokens
@torch.no_grad()
def cls_attention(modell, bild_tensor):
    """Attention vom CLS-Token auf die Patches, gemittelt über die Köpfe des letzten Blocks."""
    modell.eval()
    letzter = modell.bloecke[-1]
    letzter.attention_merken = True
    modell(bild_tensor.unsqueeze(0).to(device))
    gewichte = letzter.letzte_attention          # (1, Köpfe, L, L)
    letzter.attention_merken = False
    karte = gewichte[0, :, 0, 1:].mean(dim=0)    # CLS-Zeile ohne CLS-Spalte
    g = modell.gitter
    return karte.reshape(g, g).cpu().numpy()


beispiele = [mnist_test[i] for i in range(6)]
fig, achsen = plt.subplots(2, 6, figsize=(12.0, 4.3))
for spalte, (bild, ziel) in enumerate(beispiele):
    karte = cls_attention(vit, bild)
    with torch.no_grad():
        pred = vit(bild.unsqueeze(0).to(device)).argmax(dim=1).item()

    oben = achsen[0, spalte]
    oben.imshow(bild.squeeze(0), cmap="gray")
    oben.set_title(f"wahr {ziel} · ViT {pred}", fontsize=9,
                   color="#1A1A1A" if pred == ziel else ORANGE)
    oben.set_xticks([]); oben.set_yticks([])

    unten = achsen[1, spalte]
    unten.imshow(np.kron(karte, np.ones((PATCH, PATCH))), cmap="Blues",
                 vmin=0, vmax=karte.max())
    rohbild = (bild.squeeze(0) * 0.3081 + 0.1307).numpy()      # Normierung rückgängig
    unten.contour(rohbild, levels=[0.5], colors=ORANGE, linewidths=1.4)
    unten.set_xticks([]); unten.set_yticks([])
achsen[0, 0].set_ylabel("Bild", fontsize=9)
achsen[1, 0].set_ylabel("Attention", fontsize=9)
plt.tight_layout()
plt.show()

print("Dunkelblau = das CLS-Token holt sich von hier viel Information.")
print("Die orange Linie ist der Umriss der Ziffer.")

---
## 7 · Selbst zeichnen und beide Modelle befragen

Nur in Colab. Zeichne eine Ziffer weiss auf schwarz — beide Modelle sagen ihre Vorhersage,
und für den ViT siehst du zusätzlich, worauf er geschaut hat.

In [ ]:
#@title Ziffer zeichnen und klassifizieren (nur Colab)
if not IN_COLAB:
    print("Diese Zelle läuft nur in Google Colab.")
else:
    from google.colab.output import eval_js
    from IPython.display import display, Javascript
    from base64 import b64decode
    import io
    from PIL import Image

    js = Javascript('''
    async function draw_digit() {
      return await new Promise((resolve) => {
        const div = document.createElement('div');
        div.style.margin = '8px 0';
        const title = document.createElement('div');
        title.textContent = 'Ziffer zeichnen (weiss auf schwarz), dann auf Vorhersagen klicken.';
        title.style.margin = '6px 0';
        const canvas = document.createElement('canvas');
        canvas.width = 280; canvas.height = 280;
        canvas.style.border = '1px solid #999';
        canvas.style.touchAction = 'none';
        const ctx = canvas.getContext('2d');
        ctx.fillStyle = 'black'; ctx.fillRect(0, 0, canvas.width, canvas.height);
        ctx.strokeStyle = 'white'; ctx.lineWidth = 18;
        ctx.lineCap = 'round'; ctx.lineJoin = 'round';
        let drawing = false;
        const pos = (e) => {
          const r = canvas.getBoundingClientRect();
          const t = e.touches ? e.touches[0] : e;
          return [t.clientX - r.left, t.clientY - r.top];
        };
        canvas.addEventListener('pointerdown', (e) => {
          drawing = true; const [x, y] = pos(e); ctx.beginPath(); ctx.moveTo(x, y);
        });
        canvas.addEventListener('pointermove', (e) => {
          if (!drawing) return; const [x, y] = pos(e); ctx.lineTo(x, y); ctx.stroke();
        });
        canvas.addEventListener('pointerup', () => { drawing = false; });
        const btn = document.createElement('button');
        btn.textContent = 'Vorhersagen';
        btn.style.margin = '8px 8px 0 0';
        const clr = document.createElement('button');
        clr.textContent = 'Löschen';
        clr.onclick = () => { ctx.fillStyle = 'black'; ctx.fillRect(0, 0, 280, 280); };
        btn.onclick = () => resolve(canvas.toDataURL('image/png'));
        div.appendChild(title); div.appendChild(canvas);
        div.appendChild(document.createElement('br'));
        div.appendChild(btn); div.appendChild(clr);
        document.body.appendChild(div);
      });
    }
    ''')
    display(js)
    daten_url = eval_js('draw_digit()')
    roh = b64decode(daten_url.split(',')[1])
    bild = Image.open(io.BytesIO(roh)).convert('L').resize((28, 28), Image.LANCZOS)

    x = torch.tensor(np.array(bild), dtype=torch.float32) / 255.0
    x = ((x - 0.1307) / 0.3081).unsqueeze(0)             # (1, 28, 28)

    with torch.no_grad():
        p_cnn = F.softmax(cnn(x.unsqueeze(0).to(device)), dim=1)[0].cpu()
        p_vit = F.softmax(vit(x.unsqueeze(0).to(device)), dim=1)[0].cpu()

    karte = cls_attention(vit, x)

    fig, achsen = plt.subplots(1, 3, figsize=(10.4, 3.4))
    achsen[0].imshow(np.array(bild), cmap="gray")
    achsen[0].set_title(f"CNN: {p_cnn.argmax().item()} ({p_cnn.max():.0%})   "
                        f"ViT: {p_vit.argmax().item()} ({p_vit.max():.0%})",
                        loc="left", fontsize=10)
    achsen[0].set_xticks([]); achsen[0].set_yticks([])

    achsen[1].imshow(np.kron(karte, np.ones((7, 7))), cmap="Blues", vmin=0, vmax=karte.max())
    achsen[1].contour(np.array(bild) / 255.0, levels=[0.5], colors=ORANGE, linewidths=1.4)
    achsen[1].set_title("Attention des ViT", loc="left", fontsize=10)
    achsen[1].set_xticks([]); achsen[1].set_yticks([])

    breite = 0.38
    achsen[2].bar(np.arange(10) - breite / 2, p_cnn, breite, color=BLAU, label="CNN")
    achsen[2].bar(np.arange(10) + breite / 2, p_vit, breite, color=ORANGE, label="ViT")
    achsen[2].set_xticks(range(10)); achsen[2].set_ylim(0, 1)
    achsen[2].set_title("Wahrscheinlichkeiten", loc="left", fontsize=10)
    achsen[2].legend(frameon=False)
    plt.tight_layout()
    plt.show()

---
## 8 · Was man daraus lernt

Halte das Ergebnis aus Abschnitt 5 neben die Architekturen:

**Das CNN gewinnt hier — und das ist kein Zufall.** MNIST ist ein kleines Problem mit starker
räumlicher Struktur. Genau dafür ist die Faltung gebaut. Lokalität und Gewichtsteilung sind
Annahmen über die Welt, die hier zutreffen, und ein Modell, das richtige Annahmen mitbringt,
braucht weniger Daten, um gut zu werden.

**Der Transformer muss dieselben Annahmen erst aus den Daten lernen.** Er darf von Anfang an
jeden Patch mit jedem vergleichen — eine viel grössere Freiheit. Freiheit kostet Daten. Bei 50 000
Bildern reicht es für einen anständigen, aber nicht für einen besseren Wert.

**Die Umkehrung gilt auch, und sie ist der Grund, warum es Vision Transformer überhaupt gibt.**
Ab einigen Millionen Bildern schlagen ViTs vergleichbare CNNs, weil sie dann Beziehungen lernen
können, die eine Faltung gar nicht ausdrücken kann. Der induktive Bias, der bei wenig Daten hilft,
wird bei viel Daten zur Fessel.

**Die Lehre für die Praxis:** Nicht „welche Architektur ist die beste", sondern „welche Annahmen
über meine Daten kann ich einbauen, und habe ich genug Daten, um darauf zu verzichten".

### Zur Trainingszeit in der Tabelle

Diese Spalte sagt weniger über die Architekturen aus, als sie vorgibt. Der ViT besteht aus sehr
vielen kleinen Matrixoperationen, das CNN aus wenigen grossen. Auf einer CPU ist das für den ViT
ein schlechtes Geschäft, auf einer GPU ein gutes — das Verhältnis kann sich je nach Laufzeitumgebung
umdrehen. Vergleiche die Zeiten also innerhalb deines eigenen Laufs und lies sie nicht als
Eigenschaft der Architektur.

### Eine Ehrlichkeit zum Vergleich

Schau nochmals auf die Parameterspalte: Der ViT ist hier das **deutlich kleinere** Modell. Das
CNN trägt den Grossteil seiner Parameter in der ersten vollverbundenen Schicht
(`64·12·12 → 128`, also rund 1,2 Millionen), während der ganze Transformer mit einem Bruchteil
auskommt.

Kontrolliert ist dieser Vergleich also über **Daten und Trainingsbudget**, nicht über
Modellgrösse. Das ist eine bewusste Entscheidung — in der Praxis hat man selten gleich grosse
Modelle zur Auswahl, sondern ein festes Datenbudget. Wer es anders kontrollieren will, findet
das in Aufgabe 5.

---

### Aufgaben zum Weiterspielen

1. **Mehr Epochen für den ViT.** Setze `cfg.epochs` auf 15 und trainiere nur den ViT nach. Holt
   er auf? Ab wann flacht die Kurve ab?
2. **Patch-Grösse.** Probiere `ViT(patch=4)` — das ergibt 49 statt 16 Tokens. Was passiert mit
   Genauigkeit, Parameterzahl und Laufzeit?
3. **Weniger Daten.** Setze `cfg.train_subset = 5000` und trainiere beide neu. Wird der Abstand
   grösser oder kleiner? Das ist der direkteste Test der These aus Abschnitt 8.
4. **Positions-Embedding ausschalten.** Kommentiere `+ self.pos_embedding` aus. Um wie viel bricht
   die Genauigkeit ein — und warum trifft es den ViT, nicht das CNN?
5. **Fairer nach Modellgrösse.** Vergrössere den ViT, bis er ungefähr so viele Parameter hat wie
   das CNN — etwa `ViT(dim=128, tiefe=6, heads=8, mlp_dim=256)`. Prüfe die Parameterzahl mit
   `anzahl_parameter(...)`, bevor du trainierst. Schliesst sich die Lücke, wenn die Modelle
   gleich gross sind? Was sagt dir die Antwort über die Rolle des induktiven Bias?
6. **Mehr Tiefe statt mehr Breite.** Vergleiche `ViT(tiefe=8, dim=64)` mit `ViT(tiefe=4, dim=128)`
   bei ähnlicher Parameterzahl.